# 32 — Weather, Ground, and News Fusion

Fuses weather, optional ground data, and relevant news for Newhaven daily analysis.

In [ ]:

CENTER_NAME = "Newhaven ERF"
CENTER_LAT = 50.80208
CENTER_LON = 0.04961
DATE_FROM = "2024-01-01"
DATE_TO   = "2025-12-31"
OUTPUT_DIR = "outputs/newhaven_fusion"
GROUND_CANDIDATE_PATHS = ["outputs/ground/ground_daily.csv","outputs/ground/station_daily.csv","outputs/ground_data/ground_daily.csv","database/ground_daily.csv"]
PLACE_TERMS = ["Newhaven","Lewes","Eastbourne","Seaford","Peacehaven","Brighton","Shoreham"]
EVENT_TERMS = ["fire","smoke","industrial","plant","incinerator","pollution","air quality","traffic","port","gas leak"]
MAX_ARTICLES_PER_SOURCE = 50


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True); CACHE_DIR=Path(OUTPUT_DIR)/"weather_cache"; CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

j=fetch_weather_cached(CACHE_DIR,CENTER_LAT,CENTER_LON,DATE_FROM,DATE_TO,"wind_speed_10m,wind_direction_10m,cloud_cover,visibility,temperature_2m")
h=j.get("hourly",{})
wx=pd.DataFrame({"time_utc":pd.to_datetime(h.get("time",[]),utc=True,errors="coerce"),"wind_speed_ms":h.get("wind_speed_10m",[]),"wind_dir_deg":h.get("wind_direction_10m",[]),"cloud_cover_pct":h.get("cloud_cover",[]),"visibility_m":h.get("visibility",[]),"temperature_c":h.get("temperature_2m",[])})
wx["date"]=wx["time_utc"].dt.date.astype("string")
weather_hourly_csv=Path(OUTPUT_DIR)/"weather_hourly.csv"
wx.to_csv(weather_hourly_csv,index=False)
weather_daily=wx.groupby("date",dropna=False).agg(mean_wind_speed_ms=("wind_speed_ms","mean"),mean_cloud_cover_pct=("cloud_cover_pct","mean"),max_visibility_m=("visibility_m","max"),mean_temperature_c=("temperature_c","mean")).reset_index()


In [ ]:

ground_frames=[]; ground_meta=[]
for p in GROUND_CANDIDATE_PATHS:
    df,meta=safe_read_csv(p)
    if meta: ground_meta.append(meta)
    if not df.empty:
        df["source_path"]=p; ground_frames.append(df)
if ground_frames:
    ground=pd.concat(ground_frames,ignore_index=True,sort=False)
    if "date" in ground.columns:
        ground["date"]=pd.to_datetime(ground["date"],errors="coerce").dt.date.astype("string")
    elif "time_utc" in ground.columns:
        ground["date"]=pd.to_datetime(ground["time_utc"],utc=True,errors="coerce").dt.date.astype("string")
    ground_daily=ground.groupby("date",dropna=False).size().reset_index(name="ground_rows")
else:
    ground=pd.DataFrame(); ground_daily=pd.DataFrame(columns=["date","ground_rows"])
ground_daily_csv=Path(OUTPUT_DIR)/"ground_data_daily.csv"
ground_daily.to_csv(ground_daily_csv,index=False)


In [ ]:

NEWSAPI_KEY=os.getenv("NEWSAPI_KEY")
NEWSDATA_KEY=os.getenv("NEWSDATA_KEY") or os.getenv("NEWSDATA_IO_KEY")
QUERY_STRING=" OR ".join(PLACE_TERMS)+" AND ("+" OR ".join(EVENT_TERMS)+")"
def request_json_with_retry(url,params=None,retries=3,timeout=40):
    last_err=None
    for attempt in range(retries):
        try:
            r=requests.get(url,params=params,timeout=timeout); r.raise_for_status(); return r.json()
        except Exception as e:
            last_err=e
            if attempt<retries-1: time.sleep(2**attempt)
    raise RuntimeError(f"Request failed after {retries} attempts: {last_err}")
article_rows=[]; api_log=[]
if NEWSAPI_KEY:
    try:
        data=request_json_with_retry("https://newsapi.org/v2/everything",params={"q":QUERY_STRING,"from":DATE_FROM,"to":DATE_TO,"language":"en","pageSize":MAX_ARTICLES_PER_SOURCE,"apiKey":NEWSAPI_KEY})
        for a in data.get("articles",[]):
            article_rows.append({"source_api":"newsapi","published_utc":a.get("publishedAt"),"title":a.get("title"),"description":a.get("description"),"source_name":(a.get("source") or {}).get("name"),"url":a.get("url")})
        api_log.append({"api":"newsapi","status":"ok","rows":len(data.get("articles",[]))})
    except Exception as e: api_log.append({"api":"newsapi","status":"error","error":str(e)})
else: api_log.append({"api":"newsapi","status":"missing_key"})
if NEWSDATA_KEY:
    try:
        data=request_json_with_retry("https://newsdata.io/api/1/news",params={"apikey":NEWSDATA_KEY,"q":QUERY_STRING,"country":"gb","language":"en"})
        for a in data.get("results",[])[:MAX_ARTICLES_PER_SOURCE]:
            article_rows.append({"source_api":"newsdata","published_utc":a.get("pubDate"),"title":a.get("title"),"description":a.get("description") or a.get("content"),"source_name":a.get("source_id") or a.get("source_name"),"url":a.get("link")})
        api_log.append({"api":"newsdata","status":"ok","rows":min(len(data.get("results",[])),MAX_ARTICLES_PER_SOURCE)})
    except Exception as e: api_log.append({"api":"newsdata","status":"error","error":str(e)})
else: api_log.append({"api":"newsdata","status":"missing_key"})
articles=pd.DataFrame(article_rows)
if not articles.empty:
    articles["published_utc"]=pd.to_datetime(articles["published_utc"],utc=True,errors="coerce")
    articles["date"]=articles["published_utc"].dt.date.astype("string")
news_csv=Path(OUTPUT_DIR)/"news_articles.csv"
articles.to_csv(news_csv,index=False)
news_daily=articles.groupby("date",dropna=False).agg(article_count=("url","count"),top_title=("title","first")).reset_index() if not articles.empty else pd.DataFrame(columns=["date","article_count","top_title"])


In [ ]:

fusion=weather_daily.merge(ground_daily,on="date",how="left").merge(news_daily,on="date",how="left")
if "ground_rows" not in fusion.columns: fusion["ground_rows"]=0
if "article_count" not in fusion.columns: fusion["article_count"]=0
fusion["ground_rows"]=pd.to_numeric(fusion["ground_rows"],errors="coerce").fillna(0)
fusion["article_count"]=pd.to_numeric(fusion["article_count"],errors="coerce").fillna(0)
fusion["fusion_score"]=((5-pd.to_numeric(fusion["mean_wind_speed_ms"],errors="coerce").fillna(5)).clip(lower=0)+(100-pd.to_numeric(fusion["mean_cloud_cover_pct"],errors="coerce").fillna(100)).clip(lower=0)/25.0+fusion["ground_rows"]*0.05+fusion["article_count"]*0.5)
fusion_csv=Path(OUTPUT_DIR)/"fusion_daily.csv"
fusion.to_csv(fusion_csv,index=False)
display(fusion.head(30))


In [ ]:

fig,ax=plt.subplots(figsize=(10,5))
if not fusion.empty:
    x=pd.to_datetime(fusion["date"]); ax.plot(x,fusion["fusion_score"],marker="o",label="fusion_score"); ax.plot(x,fusion["article_count"],marker="o",label="article_count")
ax.set_title("Newhaven weather-ground-news fusion"); ax.set_xlabel("Date"); ax.set_ylabel("Score / count")
if not fusion.empty: ax.legend()
fig.autofmt_xdate(); fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/"fusion_plot.png"; fig.savefig(plot_path,dpi=150); plt.show()
manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"center_name":CENTER_NAME,"center_lat":CENTER_LAT,"center_lon":CENTER_LON,"date_from":DATE_FROM,"date_to":DATE_TO,"ground_meta":ground_meta,"api_log":api_log,"rows_weather_daily":int(len(weather_daily)),"rows_ground_daily":int(len(ground_daily)),"rows_news":int(len(articles)),"rows_fusion":int(len(fusion)),"outputs":{"weather_hourly_csv":str(weather_hourly_csv),"ground_daily_csv":str(ground_daily_csv),"news_csv":str(news_csv),"fusion_csv":str(fusion_csv),"plot_png":str(plot_path)}}
manifest_path=Path(OUTPUT_DIR)/"fusion_manifest.json"; manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Saved:",manifest_path)
